In [16]:
from mint.utils import get_file_list
from imageio import imread
from pathlib import Path
import matplotlib.pyplot as plt
from mint.denoising import wavelet, tophat, filtering
import ipywidgets as widgets
import numpy as np
import trackpy as tp

%matplotlib widget

In [17]:
plt.close()

In [18]:
gfp_folder = r'/media/lumin/DATA/Coloc/20240906_Lamelle_1/GFP/'
nv_folder = r'/media/lumin/DATA/Coloc/20240906_Lamelle_1/NV/'

In [19]:
gfp_paths, gfp_names = get_file_list(Path(gfp_folder), '.tif')

In [20]:
nv_paths, nv_names = get_file_list(Path(nv_folder), '.tif')

In [28]:
test = imread(nv_paths[3])

/tmp/ipykernel_488036/1179938247.py:1: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  test = imread(nv_paths[3])


In [24]:
nv_paths[2]

'/media/lumin/DATA/Coloc/20240906_Lamelle_1/NV/Captured iXon 885 - DIATIRF x100 NV #10.tif'

In [26]:
frame = imread(nv_paths[3])
frame = frame.astype('float64')
frame = wavelet(frame)

raw_coordinates = tp.locate(frame,
                            minmass=4000,
                            diameter=11,
                            separation=10,
                            preprocess=False,
                            engine='numba')

/tmp/ipykernel_488036/3846488676.py:1: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  frame = imread(nv_paths[3])


In [ ]:
plt.close()
plt.figure(figsize=(10, 10))
plt.imshow(test, cmap='gray')

In [12]:
filtered = test.copy()

In [ ]:
plt.close()
plt.figure(figsize=(10, 10))
plt.imshow(wavelet(filtered), cmap='gray')

In [ ]:
plt.close()
plt.figure(figsize=(10, 10))
plt.imshow(tophat(4, filtered), cmap='gray')

In [29]:
style = {'description_width': 'initial'}

minmass_lower = widgets.IntText(description='Minmass, lower bound', style=style, value=0)
minmass_upper = widgets.IntText(description='Minmass, upper bound', style=style, value=100)
diameter_lower = widgets.IntText(description='Diameter, lower bound', style=style, value=0)
diameter_upper = widgets.IntText(description='Diameter, upper bound', style=style, value=20)
separation_lower = widgets.IntText(description='Separation, lower bound', style=style, value=0)
separation_upper = widgets.IntText(description='Separation, upper bound', style=style, value=20)

minmass_range = widgets.IntSlider(description='Minmass',
                                  continuous_update=False,
                                  readout=True,
                                  min=minmass_lower.value,
                                  max=minmass_upper.value,
                                  value=np.mean([minmass_lower.value, minmass_upper.value]))

diameter_range = widgets.IntSlider(description='Diameter',
                                  continuous_update=False,
                                  readout=True,
                                  min=diameter_lower.value,
                                  max=diameter_upper.value,
                                  value=np.mean([diameter_lower.value, diameter_upper.value]))

separation_range = widgets.IntSlider(description='Separation',
                                  continuous_update=False,
                                  readout=True,
                                  min=separation_lower.value,
                                  max=separation_upper.value,
                                  value=np.mean([separation_lower.value, separation_upper.value]))



def test_locate(tophat, wavelet, minmass, diameter, separation):
    plt.close()
    if not diameter % 2:
        diameter = diameter + 1

    f_frame = test.astype('float64')

    settings = {'tophat': False,
                'wavelet': False}

    if tophat:
        settings['tophat'] = True
    if wavelet:
        settings['wavelet'] = True

    f_frame = filtering(frames=f_frame.copy(),
                        settings=settings,
                        parameters={'separation':separation})
        
    raw_coordinates = tp.locate(f_frame,
                                minmass=minmass,
                                diameter=diameter,
                                separation=separation,
                                preprocess=False,
                                engine='numba')
    plt.figure(figsize=(10, 10))
    plt.imshow(f_frame[0])
    plt.title("Locate parameters test", fontsize=10)
    plt.xlabel("x (pixels)", fontsize=10)
    plt.ylabel("y (pixels)", fontsize=10)
    plt.xticks(fontsize=10)
    plt.yticks(fontsize=10)
    tp.annotate(raw_coordinates,
                f_frame[0],
                plot_style={'markersize': 10},
                color='red')
    plt.figtext(0.5, 0.01,
                f'Number of particles found :  {len(raw_coordinates)}',
                ha='center')
    plt.show()

def update_bounds(ml, mu, dl, du, sl, su):
    minmass_range.min = ml
    minmass_range.max = mu
    diameter_range.min = dl
    diameter_range.max = du
    separation_range.min = sl
    separation_range.max = su

In [30]:
widgets.interact(update_bounds,
                 ml=minmass_lower,
                 mu=minmass_upper,
                 dl=diameter_lower,
                 du=diameter_upper,
                 sl=separation_lower,
                 su=separation_upper)

interactive(children=(IntText(value=0, description='Minmass, lower bound', style=DescriptionStyle(description_…

<function __main__.update_bounds(ml, mu, dl, du, sl, su)>

In [31]:
widgets.interact(test_locate,
                 tophat=False,
                 wavelet=False,
                 minmass=minmass_range,
                 diameter=diameter_range,
                 separation=separation_range)

interactive(children=(Checkbox(value=False, description='tophat'), Checkbox(value=False, description='wavelet'…

<function __main__.test_locate(tophat, wavelet, minmass, diameter, separation)>